# Causal expected-value policy — Memecoins: Pump or Dump?

**Francisco Angulo de Lafuente** · [ORCID0009-0001-1634-7063](https://orcid.org/0009-0001-1634-7063) · [GitHubAgnuxo1](https://github.com/Agnuxo1)

CPU only, no internet, no pretrained model. Every classifier and calibration is fitted in this run. Each token uses its own visible first minute and resolved past observations. Historical validation days are treated as wholly unlabelled batches. Thresholds use only earlier, resolved training outcomes; no test ranking or purchase cap is applied.

The optional random recurrent features use five volume/count/wallet/pressure channels, with no supervised readout. This is a classical NumPy reservoir; no optical hardware or quantum speedup is claimed. The Holographic-Reservoir source remains attributed under Apache-2.0. The policy retains the fixed fit-segment classifiers after temporal calibration.

This is competition research under the organiser's idealised PnL, not evidence of real-world trading returns. No official result or superiority is claimed by this source export. The 1–14December forward rotations decide prizes, subject to the official rules. Source and notebook: Apache-2.0.

[Competition and current mechanics](https://www.kaggle.com/competitions/memecoins-pump-or-dump/overview) · [Rules](https://www.kaggle.com/competitions/memecoins-pump-or-dump/rules)


In [ ]:
import os
for _key in ('OMP_NUM_THREADS','MKL_NUM_THREADS','OPENBLAS_NUM_THREADS','NUMEXPR_NUM_THREADS'): os.environ[_key]='4'


In [ ]:
# Feature and random-reservoir functions preserved from the prepared solution.
# Francisco Angulo de Lafuente. Holographic-Reservoir original: Apache-2.0.
# No pretrained parameters; fit_readout is retained for provenance but NEVER USED.
import hashlib
import numpy as np
import pandas as pd
WSOL='So11111111111111111111111111111111111111112'
WIN=3600
NBIN,BINW,NCH=20,3,7
HRC_SIZE=48


def gini(v):
    v = np.sort(np.asarray(v, float))
    n = len(v)
    s = v.sum()
    if n == 0 or s <= 0:
        return np.nan
    return float((2 * np.arange(1, n + 1) - n - 1).dot(v) / (n * s))


def curve_feats(craw):
    """Fase de curva. Las filas con failed != 0 son intentos revertidos: se excluyen
    de todo recuento de volumen o precio y se cuentan aparte."""
    g0 = craw.groupby('mint', sort=False)
    FF = pd.DataFrame({'c_nfail': craw[craw.failed != 0].groupby('mint').size(),
                       'c_failshare': craw[craw.failed != 0].groupby('mint').size() / g0.size()})
    c = craw[craw.failed == 0]
    if len(c) == 0:
        return FF
    c = c.copy()
    c['is_buy'] = (c.direction == 'buy')     # la curva usa buy/sell, siempre con SOL de quote
    g = c.groupby('mint', sort=False)
    F = pd.DataFrame(index=g.size().index)
    F['c_n'] = g.size()
    F['c_uw'] = g.wallet.nunique()
    F['c_tpw'] = F.c_n / F.c_uw.clip(lower=1)
    F['c_sol'] = g.sol.sum()
    b = c[c.is_buy]
    s = c[~c.is_buy]
    F['c_buy_sol'] = b.groupby('mint').sol.sum()
    F['c_sell_sol'] = s.groupby('mint').sol.sum()
    F['c_sell_share'] = F.c_sell_sol / (F.c_buy_sol.fillna(0) + F.c_sell_sol.fillna(0)).replace(0, np.nan)
    F['c_dur'] = -g.ts_rel.min()
    F['c_last10_share'] = c[c.ts_rel >= -600].groupby('mint').sol.sum() / F.c_sol.replace(0, np.nan)
    F['c_last60_share'] = c[c.ts_rel >= -60].groupby('mint').sol.sum() / F.c_sol.replace(0, np.nan)
    ws = b.groupby(['mint', 'wallet']).sol.sum()
    tot = ws.groupby('mint').sum().replace(0, np.nan)
    F['c_top1'] = ws.groupby('mint').max() / tot
    F['c_top3'] = ws.groupby('mint').apply(lambda x: np.sort(x.values)[-3:].sum()) / tot
    F['c_gini'] = ws.groupby('mint').apply(lambda x: gini(x.values))
    F['c_maxbuy'] = b.groupby('mint').sol.max()
    F['c_feepayer_diff'] = g.apply(lambda x: (x.fee_payer != x.wallet).mean())
    bw = b.groupby('mint').wallet.apply(set)
    sw = s.groupby('mint').wallet.apply(set)
    F['c_roundtrip'] = pd.Series({m: len(v & sw.get(m, set())) / max(len(v), 1) for m, v in bw.items()})
    F['c_slotclust'] = g.apply(lambda x: 1 - x.slot.nunique() / len(x))
    F['c_price_last'] = g.new_price.last()
    F['c_price_max'] = g.new_price.max()
    return F.join(FF, how='outer')


def amm_prep(a):
    a = a.copy()
    raw_buy = a.direction.astype(str).str.upper().str.startswith('B')
    # direction es relativa a base_token: si SOL es la base, B es vender la moneda
    a['is_buy'] = np.where(a.base_token == WSOL, ~raw_buy, raw_buy)
    a['wallet'] = a.signing_wallet
    # el precio sale de las reservas del pool, nunca del cociente de la propia operacion
    a['sol_res'] = np.where(a.base_token == WSOL, a.pool_base_token_reserves_after,
                            a.pool_quote_token_reserves_after) / 1e9
    return a


def amm_feats(a):
    """Los primeros 60 s en el pool, separados en subventanas 0-15 / 15-30 / 30-60."""
    g = a.groupby('mint', sort=False)
    F = pd.DataFrame(index=g.size().index)
    F['a_n'] = g.size()
    F['a_uw'] = g.wallet.nunique()
    F['a_tpw'] = F.a_n / F.a_uw.clip(lower=1)
    F['a_sol'] = g.sol.sum()
    b = a[a.is_buy]
    F['a_buy_sol'] = b.groupby('mint').sol.sum()
    F['a_buyshare'] = F.a_buy_sol / F.a_sol.replace(0, np.nan)
    F['a_maxbuy'] = b.groupby('mint').sol.max()
    for lo, hi, tag in [(0, 15, 'w1'), (15, 30, 'w2'), (30, 61, 'w3')]:
        w = a[(a.ts_rel >= lo) & (a.ts_rel < hi)]
        F['a_sol_' + tag] = w.groupby('mint').sol.sum()
        F['a_n_' + tag] = w.groupby('mint').size()
        F['a_buyshare_' + tag] = w[w.is_buy].groupby('mint').sol.sum() / F['a_sol_' + tag].replace(0, np.nan)
    F['a_mom'] = F.a_sol_w3 / F.a_sol_w1.replace(0, np.nan)
    F['a_mom_n'] = F.a_n_w3 / F.a_n_w1.replace(0, np.nan)
    ws = b.groupby(['mint', 'wallet']).sol.sum()
    tot = ws.groupby('mint').sum().replace(0, np.nan)
    F['a_top1'] = ws.groupby('mint').max() / tot
    F['a_gini'] = ws.groupby('mint').apply(lambda x: gini(x.values))
    F['a_p_first'] = g.new_price.first()
    F['a_p_last'] = g.new_price.last()
    F['a_p_max'] = g.new_price.max()
    F['a_p_min'] = g.new_price.min()
    F['a_ret'] = F.a_p_last / F.a_p_first.replace(0, np.nan)
    F['a_runup'] = F.a_p_max / F.a_p_first.replace(0, np.nan)
    F['a_dd'] = F.a_p_last / F.a_p_max.replace(0, np.nan)
    F['a_depth_last'] = g.sol_res.last()
    F['a_depth_first'] = g.sol_res.first()
    F['a_depth_growth'] = F.a_depth_last / F.a_depth_first.replace(0, np.nan)
    F['a_fee'] = g.fee.mean()
    F['a_feepayer_diff'] = g.apply(lambda x: (x.fee_payer != x.signing_wallet).mean())
    F['a_slotclust'] = g.apply(lambda x: 1 - x.slot.nunique() / len(x))
    early = a[a.ts_rel < 15]
    late = a[a.ts_rel >= 45]
    seen = early.groupby('mint').wallet.apply(set)
    lw = late.groupby('mint').wallet.apply(set)
    F['a_fresh_late'] = pd.Series({m: len(v - seen.get(m, set())) for m, v in lw.items()})
    eb = early[early.is_buy].groupby('mint').wallet.apply(set)
    sl = a[~a.is_buy].groupby('mint').wallet.apply(set)
    F['a_flip'] = pd.Series({m: len(v & sl.get(m, set())) / max(len(v), 1) for m, v in eb.items()})
    return F


def amm_seq(a):
    """Secuencia binada del primer minuto, (NBIN, NCH). Es lo que lee el reservorio.
    Canales: SOL comprado, SOL vendido, n operaciones, carteras nuevas,
    log precio relativo al primero, log profundidad relativa, cuota de compra."""
    if not a.ts_rel.between(0,60,inclusive='both').all():
        raise ValueError('AMM sequence contains a trade outside the visible first minute')
    a = a.copy()
    a['b'] = np.clip(a.ts_rel.values // BINW, 0, NBIN - 1).astype(int)
    a = a.sort_values(['mint', 'ts_rel'], kind='stable')
    mints = a.mint.unique()
    idx = {m: i for i, m in enumerate(mints)}
    T = np.zeros((len(mints), NBIN, NCH), dtype=np.float32)
    mi = a.mint.map(idx).values
    bi = a.b.values
    sol = a.sol.values.astype(np.float64)
    isb = a.is_buy.values.astype(bool)
    np.add.at(T, (mi[isb], bi[isb], 0), sol[isb])
    np.add.at(T, (mi[~isb], bi[~isb], 1), sol[~isb])
    np.add.at(T, (mi, bi, 2), 1.0)
    p0 = a.groupby('mint').new_price.first().reindex(mints).values
    d0 = a.groupby('mint').sol_res.first().reindex(mints).values
    last = a.groupby(['mint', 'b']).agg(p=('new_price', 'last'), d=('sol_res', 'last')).reset_index()
    li = last.mint.map(idx).values
    lb = last.b.values
    T[li, lb, 4] = np.log(np.clip(last.p.values / np.clip(p0[li], 1e-18, None), 1e-6, 1e6))
    T[li, lb, 5] = np.log(np.clip(last.d.values / np.clip(d0[li], 1e-12, None), 1e-6, 1e6))
    for ch in (4, 5):
        for t in range(1, NBIN):
            m = T[:, t, 2] == 0
            T[m, t, ch] = T[m, t - 1, ch]
    tot = T[:, :, 0] + T[:, :, 1]
    T[:, :, 6] = np.where(tot > 0, T[:, :, 0] / np.maximum(tot, 1e-9), 0.0)
    nw = np.zeros((len(mints), NBIN), dtype=np.float32)
    for m, g in a.groupby('mint', sort=False):
        s = set()
        i = idx[m]
        for b_, gg in g.groupby('b', sort=True):
            new = set(gg.wallet) - s
            nw[i, b_] = len(new)
            s |= new
    T[:, :, 3] = nw
    for ch in (0, 1, 2, 3):
        T[:, :, ch] = np.log1p(T[:, :, ch])
    return mints, T


def creator_rep(a, levels=(3.0, 5.0, 10.0)):
    """Historial del creador, estrictamente acumulativo. Para el token i solo entran
    tokens con mt <= mt_i - WIN, es decir cuya propia hora ya cerro. Nunca se usa el
    fichero completo: borrar tokens posteriores no cambia ningun valor."""
    mts, mx = a.mt.values, a.max_x.values
    cre = a.creator.fillna('?').astype('category').cat.codes.values
    nC = cre.max() + 1
    n = np.zeros(nC)
    hits = {u: np.zeros(nC) for u in levels}
    out = np.full((len(a), 1 + len(levels)), np.nan)
    j = 0
    for i in range(len(a)):
        while j < len(a) and mts[j] <= mts[i] - WIN:
            if not np.isnan(mx[j]):
                n[cre[j]] += 1
                for u in levels:
                    if mx[j] >= u:
                        hits[u][cre[j]] += 1
            j += 1
        c = cre[i]
        k = n[c]
        out[i, 0] = k
        for q, u in enumerate(levels):
            out[i, 1 + q] = hits[u][c] / k if k > 0 else np.nan
    cols = ['r_creator_n'] + ['r_creator_p%g' % u for u in levels]
    return pd.DataFrame(out, columns=cols, index=a.index)


def wallet_rep(a, E, k_level=5.0, min_seen=3):
    """Reputacion de los compradores del primer minuto, con la misma regla de
    resolucion. Un token que migro hace 20 minutos no ha resuelto y no cuenta."""
    cols = ['r_w_known', 'r_w_max', 'r_w_mean', 'r_w_top3', 'r_w_elite_n',
            'r_w_elite_share', 'r_w_nwal']
    if E is None or len(E) == 0:
        return pd.DataFrame(np.nan, columns=cols, index=a.index)
    E = E.copy()
    E['wid'] = E.wallet.astype('category').cat.codes.astype(np.int64)
    by = {m: (g.wid.values, g.sol.values)
          for m, g in E.sort_values('mint', kind='stable').groupby('mint', sort=False)}
    nW = int(E.wid.max()) + 1
    seen = np.zeros(nW, np.int32)
    hit = np.zeros(nW, np.int32)
    mts, mx, mints = a.mt.values, a.max_x.values, a.mint.values
    out = np.full((len(a), len(cols)), np.nan)
    j = 0
    for i in range(len(a)):
        while j < len(a) and mts[j] <= mts[i] - WIN:
            if not np.isnan(mx[j]):
                w = by.get(mints[j])
                if w is not None:
                    np.add.at(seen, w[0], 1)
                    if mx[j] >= k_level:
                        np.add.at(hit, w[0], 1)
            j += 1
        w = by.get(mints[i])
        if w is None:
            continue
        wid, sol = w
        out[i, 6] = len(wid)
        s = seen[wid]
        kn = s >= min_seen
        if kn.sum() == 0:
            out[i, :6] = [0, 0, 0, 0, 0, 0]
            continue
        rate = hit[wid][kn] / s[kn]
        elite = (s >= 5) & (hit[wid] >= 1)
        out[i, :6] = [kn.sum(), rate.max(), rate.mean(), np.sort(rate)[-3:].mean(),
                      elite.sum(), sol[elite].sum() / max(sol.sum(), 1e-9)]
    return pd.DataFrame(out, columns=cols, index=a.index)


def _sha_entropy(n, seed):
    """La entropia determinista del backend simulado del Holographic-Reservoir
    original (SHA-256 encadenado). Sin red y sin hardware: solo la biblioteca estandar."""
    out = []
    h = hashlib.sha256(('hrc-init-%d' % seed).encode()).digest()
    while len(out) < n:
        out.extend(h)
        h = hashlib.sha256(h).digest()
    return np.frombuffer(bytes(out[:n]), dtype=np.uint8).astype(np.float64) / 127.5 - 1.0


class HolographicReservoir:
    """Port en linea del nucleo de Holographic-Reservoir (Apache-2.0, F. Angulo):
    capa recurrente dispersa con fuga y tanh, radio espectral fijado, entropia
    SHA-256 inyectada en la diagonal, y lectura ridge (la forma cerrada de la RLS
    del original con factor de olvido 1).

    La caja de arena del concurso no tiene red, asi que el paquete no se puede
    instalar: se porta el mismo principio dinamico a NumPy puro, vectorizado
    sobre todos los tokens a la vez (un paso de tiempo para los N tokens a la vez
    en lugar de un bucle por token). Nada se entrena fuera de la ejecucion: los
    pesos del reservorio son aleatorios con semilla y la lectura se ajusta aqui.
    """

    def __init__(self, size=HRC_SIZE, input_dim=NCH, spectral_radius=0.9, leak=0.3,
                 density=0.1, ridge=1e-3, seed=0):
        rng = np.random.default_rng(seed)
        self.size, self.leak, self.ridge = size, leak, ridge
        self.W_in = rng.uniform(-0.5, 0.5, size=(size, input_dim))
        W = rng.standard_normal((size, size)) * (rng.random((size, size)) < density)
        np.fill_diagonal(W, np.diag(W) + _sha_entropy(size, seed) * 0.05)
        r = np.max(np.abs(np.linalg.eigvals(W)))
        self.W = W * (spectral_radius / r) if r > 0 else W
        self.W_out = None

    def states(self, U):
        """U: (N, T, input_dim) -> (N, 2*size+1). Estado final y estado medio: el
        final lleva el instante de la decision, el medio resume la trayectoria."""
        N, T, _ = U.shape
        X = np.zeros((N, self.size))
        acc = np.zeros((N, self.size))
        for t in range(T):
            X = (1.0 - self.leak) * X + self.leak * np.tanh(U[:, t, :] @ self.W_in.T + X @ self.W.T)
            acc += X
        return np.hstack([X, acc / T, np.ones((N, 1))])

    def fit_readout(self, S, y):
        A = S.T @ S + self.ridge * np.eye(S.shape[1])
        self.W_out = np.linalg.solve(A, S.T @ y)
        return self

    def predict(self, S):
        return S @ self.W_out


In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression

LEVELS=np.array([3.,4.,5.,7.,10.,15.,20.])

SEEDS=(0,1,2)

GRID=np.round(np.arange(0.,2.51,.1),2)

WIN=3600

EMBARGO=3600

LGB=dict(n_estimators=400,learning_rate=.04,num_leaves=31,min_child_samples=60,
         subsample=.85,subsample_freq=1,colsample_bytree=.7,reg_lambda=5.,
         verbosity=-1,n_jobs=4,deterministic=True,force_col_wise=True)

SKIP={'mint','mt','max_x','final_x','day','created_ts','creator'}

REP_PREFIX='r_'

def pnl(tp,mx,fx):
    tp=np.asarray(tp,float); mx=np.asarray(mx,float); fx=np.asarray(fx,float)
    if not np.isfinite(tp).all() or not ((tp==0)|((tp>=3)&(tp<=100))).all():
        raise ValueError('Illegal prediction')
    return np.where(tp==0,0.,np.where(mx>=tp,tp-1.3,np.maximum(fx-1.3,-1.)))

class CausalPanel:
    def __init__(self,panel,edges,T):
        self.a=panel.reset_index(drop=True).copy()
        if self.a.mint.duplicated().any() or not self.a.mt.is_monotonic_increasing:
            raise ValueError('Panel must have unique mints and stable chronological order')
        if T.shape!=(len(self.a),20,7) or not np.isfinite(T).all(): raise ValueError('Bad sequence alignment/values')
        self.T=np.asarray(T,np.float32)
        self.base=[c for c in self.a if c not in SKIP and not c.startswith(REP_PREFIX)
                   and self.a[c].dtype.kind in 'fiub']
        self.cre=self.a.creator.fillna('?').astype('category').cat.codes.to_numpy()
        self.by=[None]*len(self.a); self.nw=0
        if edges is not None and len(edges):
            # Factorise identities once, not once per daily view; IDs never enter X.
            wid=pd.factorize(edges.wallet,sort=False)[0].astype(np.int32)
            mi=pd.Index(self.a.mint).get_indexer(edges.mint)
            sol=edges.sol.to_numpy(dtype=float)
            keep=mi>=0; mi=mi[keep]; wid=wid[keep]; sol=sol[keep]
            order=np.argsort(mi,kind='stable');mi=mi[order];wid=wid[order];sol=sol[order]
            if len(mi):
                ids,starts=np.unique(mi,return_index=True); ends=np.r_[starts[1:],len(mi)]
                for i,b,e in zip(ids,starts,ends):
                    if len(np.unique(wid[b:e]))!=e-b: raise ValueError('Duplicate mint/wallet edge')
                    self.by[int(i)]=(wid[b:e],sol[b:e])
                self.nw=int(wid.max())+1
        self.rep=self.reputation()
        self.train=self.a.drop(columns=[c for c in self.a if c.startswith(REP_PREFIX)])
        self.train=pd.concat([self.train,self.rep],axis=1)
        self.fe=self.base+list(self.rep.columns)
        self.day_cache={}
        # Fixed random recurrent states: no readout fitting, labels or global scaling.
        self.state=np.zeros((len(self.a),96),dtype=np.float64)
        for s in SEEDS:
            self.state+=HolographicReservoir(size=48,input_dim=5,seed=s).states(self.T[:,:,[0,1,2,3,6]])[:,:96]/len(SEEDS)
        if not np.isfinite(self.state).all(): raise ValueError('Nonfinite reservoir state')

    def reputation(self,cutoff=None):
        """Same statistics as original functions, with pre-indexed wallet edges."""
        a=self.a; mts=a.mt.to_numpy(); mx=a.max_x.to_numpy()
        valid=np.isfinite(mx)
        if cutoff is not None: valid &= mts<=cutoff
        nc=int(self.cre.max())+1
        n=np.zeros(nc); hits=np.zeros((nc,3))
        seen=np.zeros(self.nw,np.int32); whits=np.zeros(self.nw,np.int32)
        out=np.full((len(a),11),np.nan); j=0
        for i in range(len(a)):
            while j<len(a) and mts[j]<=mts[i]-WIN:
                if valid[j]:
                    c=self.cre[j];n[c]+=1;hits[c]+=(mx[j]>=np.array([3.,5.,10.]))
                    w=self.by[j]
                    if w is not None:
                        seen[w[0]]+=1
                        if mx[j]>=5: whits[w[0]]+=1
                j+=1
            c=self.cre[i];k=n[c];out[i,0]=k
            if k>0: out[i,1:4]=hits[c]/k
            w=self.by[i]
            if w is None: continue
            wid,sol=w;s=seen[wid];kn=s>=3;out[i,10]=len(wid)
            if not kn.any(): out[i,4:10]=0.;continue
            rate=whits[wid][kn]/s[kn];elite=(s>=5)&(whits[wid]>=1)
            out[i,4:10]=[kn.sum(),rate.max(),rate.mean(),np.sort(rate)[-3:].mean(),
                         elite.sum(),sol[elite].sum()/max(sol.sum(),1e-9)]
        cols=['r_creator_n','r_creator_p3','r_creator_p5','r_creator_p10','r_w_known',
              'r_w_max','r_w_mean','r_w_top3','r_w_elite_n','r_w_elite_share','r_w_nwal']
        return pd.DataFrame(out,columns=cols,index=a.index)

    def day_view(self,day):
        if day not in self.day_cache:
            mask=(self.a.day==day).to_numpy()
            if not mask.any(): raise ValueError('Day absent')
            cutoff=float(self.a.loc[mask,'mt'].min())-WIN-EMBARGO
            rep=self.reputation(cutoff)
            df=self.a.loc[mask].drop(columns=[c for c in self.a if c.startswith(REP_PREFIX)]).copy()
            for c in rep: df[c]=rep.loc[mask,c].to_numpy()
            self.day_cache[day]=df
        return self.day_cache[day].copy()

    def matrix(self,df,arm):
        X=df[self.fe].to_numpy(dtype=float,copy=True)
        X[~np.isfinite(X)]=np.nan
        if arm=='reservoir': X=np.concatenate([X,self.state[df.index.to_numpy()]],axis=1)
        elif arm!='control': raise ValueError('Unknown arm')
        return X

def predict_day(data,day,arm):
    val=data.day_view(day)
    cutoff=float(val.mt.min())-WIN-EMBARGO
    tr=data.train[data.train.max_x.notna() & (data.train.mt<=cutoff)].copy()
    days=sorted(tr.day.unique())
    if len(days)<12 or len(tr)<3000: raise ValueError('Insufficient training history')
    caldays=days[-5:]
    cal=pd.concat([data.day_view(d) for d in caldays]).loc[lambda x:x.index.isin(tr.index)]
    fit=tr[(tr.day<caldays[0]) & (tr.mt<=float(cal.mt.min())-WIN-EMBARGO)]
    if len(fit)<2000 or len(cal)<200: raise ValueError('Insufficient disjoint calibration')
    if float(fit.mt.max())>float(cal.mt.min())-WIN-EMBARGO: raise RuntimeError('Fit/calibration overlap')
    if float(tr.mt.max())>float(val.mt.min())-WIN-EMBARGO: raise RuntimeError('Unresolved labels')
    Xf=data.matrix(fit,arm); Xc=data.matrix(cal,arm); Xv=data.matrix(val,arm)
    P=np.zeros((len(val),len(LEVELS)))
    for k,u in enumerate(LEVELS):
        yf=(fit.max_x.to_numpy()>=u).astype(int)
        yc=(cal.max_x.to_numpy()>=u).astype(int)
        if yf.min()==yf.max():
            pv=np.full(len(val),float(yf[0])); pc=np.full(len(cal),float(yf[0]))
        else:
            pc=np.zeros(len(cal)); pv=np.zeros(len(val))
            for seed in SEEDS:
                model=lgb.LGBMClassifier(random_state=seed,**LGB).fit(Xf,yf)
                pc+=model.predict_proba(Xc)[:,1]/len(SEEDS)
                pv+=model.predict_proba(Xv)[:,1]/len(SEEDS)
        if np.unique(pc).size>1 and 0<yc.sum()<len(yc):
            ir=IsotonicRegression(out_of_bounds='clip',y_min=0.,y_max=1.).fit(pc,yc)
            pv=ir.transform(pv)
        else:
            # Constant calibration target is mapped to its empirical probability.
            pv=np.full(len(val),float(yc.mean()))
        P[:,k]=pv
    P=np.minimum.accumulate(P,axis=1)
    L=np.array([np.maximum(tr.loc[tr.max_x<u,'final_x'].to_numpy()-1.3,-1.).mean()
                if (tr.max_x<u).any() else -1. for u in LEVELS])
    EV=P*(LEVELS-1.3)+(1-P)*L
    k=EV.argmax(axis=1); ev=EV[np.arange(len(val)),k]; lev=LEVELS[k]
    if not np.isfinite(P).all() or not np.isfinite(ev).all(): raise RuntimeError('Bad model output')
    return {'mint':val.mint.to_numpy(dtype=str),'day':np.array([day]*len(val)),
            'mt':val.mt.to_numpy(),'ev':ev,'level':lev,'max_x':val.max_x.to_numpy(),
            'final_x':val.final_x.to_numpy(),'fit_rows':np.array([len(fit)]),
            'cal_rows':np.array([len(cal)]),'train_max_mt':np.array([tr.mt.max()]),
            'val_min_mt':np.array([val.mt.min()]),'probability':P}

def choose_tau(history,*,available_before):
    if len(history)<4: return 1e9,[]
    table=[]
    for t in GRID:
        scores=[]; buys=[]
        for x in history[-8:]:
            resolved=x['mt']<=available_before
            tp=np.where(x['ev'][resolved]>=t,x['level'][resolved],0.)
            scores.append(float(pnl(tp,x['max_x'][resolved],x['final_x'][resolved]).sum()))
            buys.append(int((tp>0).sum()))
        mean=float(np.mean(scores)); sd=float(np.std(scores,ddof=1)); depth=float(np.mean(buys))
        table.append({'tau':float(t),'mean':mean,'sd':sd,'depth':depth,'score':mean/(sd+1)})
    band=[x for x in table if .1<=x['depth']<=2. and x['mean']>0]
    return (max(band,key=lambda x:(x['score'],x['tau']))['tau'] if band else 1e9),table


In [ ]:
"""Build the same input representation from attached organiser files only.

No network, credential access, pretrained objects or external feature caches.
Francisco Angulo de Lafuente. ORCID 0009-0001-1634-7063.
"""
import gc
from pathlib import Path
import numpy as np
import pandas as pd

VISIBLE_CURVE_COLUMNS=['mint','ts_rel','direction','failed','sol','new_price','wallet','fee_payer','slot']
VISIBLE_AMM_COLUMNS=['mint','ts_rel','direction','sol','new_price','signing_wallet','fee_payer','slot','base_token',
                     'pool_base_token_reserves_after','pool_quote_token_reserves_after','fee']
CREATION_NUMERIC=['bundle_size','gas_used','amount_of_instructions','amount_of_lookup_reads','amount_of_lookup_writes',
                 'bundled_buys','bundled_buys_count','dev_balance','creation_ix_index','pf_program_index',
                 'direct_pf_invocation','version','mayhem_mode','is_cashback_enabled']

def attached_tape_path(data,day,table):
    paths=[data/('day_'+day+'_'+table+'_all_swaps.parquet'),
           data/'trades'/('day='+day)/(table+'_all_swaps.parquet')]
    for p in paths:
        if p.is_file():return p
    raise FileNotFoundError('Attached first-minute tape absent for '+day+' '+table)

def build_attached(data):
    data=Path(data)
    train_parts=[pd.read_csv(data/'train.csv')]
    if (data/'train_archive.csv').is_file():train_parts.insert(0,pd.read_csv(data/'train_archive.csv'))
    tr=pd.concat(train_parts,ignore_index=True).rename(columns={'finalx':'final_x'})
    test=pd.read_csv(data/'test.csv');sample=pd.read_csv(data/'sample_submission.csv')
    if tr.mint.duplicated().any() or test.mint.duplicated().any() or sample.mint.duplicated().any():
        raise ValueError('Duplicate organiser identity')
    if set(tr.mint)&set(test.mint) or set(sample.mint)!=set(test.mint):raise ValueError('Train/test/sample identities differ')
    if not np.isfinite(tr[['mt','max_x','final_x']]).all().all():raise ValueError('Nonfinite training target')
    test['max_x']=np.nan;test['final_x']=np.nan
    a=pd.concat([tr,test],ignore_index=True)
    a['day']=pd.to_datetime(a.mt,unit='s').dt.strftime('%Y-%m-%d')
    a=a.sort_values('mt',kind='stable').reset_index(drop=True)
    test_days=pd.to_datetime(test.mt,unit='s').dt.strftime('%Y-%m-%d').unique()
    if len(test_days)!=1:raise ValueError('Expected one test calendar day')
    creation=pd.read_csv(data/'pumpfun_token_creation.csv').drop_duplicates('mint')
    c=creation[['mint','created_ts','creator']].copy()
    for k in CREATION_NUMERIC:
        if k not in creation:raise ValueError('Creation field absent '+k)
        c['x_'+k]=pd.to_numeric(creation[k],errors='coerce')
    c['x_token2022']=(~creation.token_program.fillna('').astype(str).str.startswith('Tokenkeg')).astype(int)
    c['x_bundle_struct_n']=creation.bundle_structure.fillna('').astype(str).str.count(',')+1
    c['x_symbol_len']=creation.symbol.fillna('').astype(str).str.len()
    c['x_name_len']=creation.name.fillna('').astype(str).str.len()
    del creation
    fparts=[];eparts=[];mparts=[];tparts=[]
    for day in sorted(a.day.unique()):
        curve=pd.read_parquet(attached_tape_path(data,day,'pumpfun'),columns=VISIBLE_CURVE_COLUMNS)
        if not (curve.ts_rel<=60).all():raise ValueError('Curve exceeds visible decision time')
        amm=amm_prep(pd.read_parquet(attached_tape_path(data,day,'pumpswap'),columns=VISIBLE_AMM_COLUMNS))
        if not amm.ts_rel.between(0,60,inclusive='both').all():raise ValueError('AMM exceeds visible first minute')
        f=curve_feats(curve).join(amm_feats(amm),how='outer');f.index.name='mint'
        fparts.append(f.reset_index())
        m,t=amm_seq(amm);mparts.append(np.asarray(m,dtype=str));tparts.append(t)
        buys=amm[amm.is_buy];eparts.append(buys.groupby(['mint','wallet'],as_index=False).sol.sum())
        del curve,amm,f,m,t,buys;gc.collect()
        print('Attached data extracted:',day,flush=True)
    f=pd.concat(fparts,ignore_index=True)
    if f.mint.duplicated().any():raise ValueError('Repeated tape identity')
    panel=a.merge(f,on='mint',how='left',validate='one_to_one').merge(c,on='mint',how='left',validate='one_to_one')
    if not panel.mint.equals(a.mint) or panel.c_n.isna().any() or panel.creator.isna().any():raise ValueError('Incomplete attached panel')
    panel['x_age_s']=panel.mt-panel.created_ts;dt=pd.to_datetime(panel.mt,unit='s')
    panel['x_hour']=dt.dt.hour;panel['x_dow']=dt.dt.day_of_week
    mints=np.concatenate(mparts);tensor=np.concatenate(tparts)
    if len(np.unique(mints))!=len(mints):raise ValueError('Duplicate sequence identity')
    ix=pd.Index(mints).get_indexer(panel.mint)
    if (ix<0).any():raise ValueError('Missing sequence identity')
    tensor=tensor[ix]
    edges=pd.concat(eparts,ignore_index=True)
    if edges.duplicated(['mint','wallet']).any():raise ValueError('Duplicate wallet edge')
    return panel,edges,tensor,sample,str(test_days[0])


In [ ]:
"""Use the exact reviewed model/threshold functions to score one unlabelled day."""
import json
import os
import time
from pathlib import Path

def write_scoring_submission(pred,tau,sample,output):
    tp=np.where(pred['ev']>=tau,pred['level'],0.).astype(float)
    if len(set(pred['mint']))!=len(pred['mint']):raise ValueError('Duplicate prediction identity')
    mapping=pd.Series(tp,index=pred['mint'])
    sub=sample[['mint']].copy();sub['tp']=sub.mint.map(mapping)
    if len(sub)!=len(sample) or not sub.mint.equals(sample.mint):raise ValueError('Submission order/length differs')
    if sub.tp.isna().any() or not np.isfinite(sub.tp).all() or not ((sub.tp==0)|sub.tp.between(3,100,inclusive='both')).all():
        raise ValueError('Invalid or missing legal prediction')
    output=Path(output);tmp=output.with_suffix(output.suffix+'.tmp')
    sub.to_csv(tmp,index=False);os.replace(tmp,output)
    return sub

def score_attached(data,arm,output='submission.csv'):
    if arm not in ('control','reservoir'):raise ValueError('Unknown fixed arm')
    started=time.monotonic()
    a,e,T,sample,test_day=build_attached(data)
    panel=CausalPanel(a,e,T)
    days=sorted(a.loc[a.max_x.notna()&(a.day<test_day),'day'].unique())[-8:]
    if len(days)<4:raise ValueError('Insufficient threshold history')
    history=[predict_day(panel,day,arm)for day in days]
    current=predict_day(panel,test_day,arm)
    cutoff=float(current['val_min_mt'][0])-WIN-EMBARGO
    tau,_=choose_tau(history,available_before=cutoff)
    sub=write_scoring_submission(current,tau,sample,output)
    print(json.dumps({'test_day':test_day,'arm':arm,'rows':len(sub),'buys':int((sub.tp>0).sum()),
                     'tau':tau,'elapsed_seconds':time.monotonic()-started,'trained_inside_this_run':True,
                     'network_used':False,'gpu_used':False}),flush=True)
    return sub


In [ ]:
DATA_CANDIDATES=[Path('/kaggle/input/competitions/memecoins-pump-or-dump'),Path('/kaggle/input/memecoins-pump-or-dump'),Path('data')]
DATA=next((x for x in DATA_CANDIDATES if (x/'train.csv').is_file()),None)
if DATA is None: raise FileNotFoundError('Attach the official competition data')
ARM='control'
score_attached(DATA,ARM,Path('submission.csv'))
